# Career Pilot — Evaluation Notebook

**What this is:** An empirical evaluation of the AI matching layer in Career Pilot — a tool that scores how well a job description fits a candidate's CV and career context, using Google Gemini.

**Why it matters:** AI-based hiring filters are only useful if they're accurate, reliable, and fair. This notebook measures all three — starting from the same psychometric evaluation framework used in professional assessment science.

**Structure:**
1. Data & setup — 24 labeled JDs (Track A)
2. Baseline results — accuracy, ranking, calibration
3. Error analysis — where and why the AI fails
4. Reliability & robustness — consistency across runs and CV phrasings
5. Discrimination — does the AI correctly reject a wrong-domain candidate?
6. Prompt engineering — what we tried to improve accuracy and why it didn't work
7. Structured matching (Phase 1) — a deterministic pre-screening layer we built and evaluated
8. What's next — roadmap for Track B (predictive validity) and improved prompts

**Data:** `labels.json` — 32 job descriptions (24 original + 8 added to strengthen negative class), human relevance labels (0–3), AI scores from the V1 prompt.  
**Methodology:** `docs/EVALUATION_METHODOLOGY.md`  
**Project plan:** `docs/PROJECT_PLAN.md`


---
## Part 1 — Data & Setup

**Track A dataset:** 32 job descriptions (24 original + 8 added after initial analysis to balance the negative class) of the candidate's application history (no contamination from past decisions). Each JD was labeled blind — before seeing the AI scores — on a 0–3 relevance scale:

| Score | Meaning |
|-------|---------|
| 0 | Completely irrelevant domain |
| 1 | Adjacent — would not apply |
| 2 | Relevant — would apply with some reservations |
| 3 | Strong fit — would apply with confidence |

**Binary threshold for accuracy metrics:** human ≥ 2 = relevant, AI ≥ 50 = relevant.

**V1 prompt (the baseline):** A simple Gemini call without a structured scoring rubric — the original `batch_analyze.py` output. This is our floor baseline. The current app prompt (Option B rubric with weighted domains) hasn't been run on this dataset yet — that's P1-revised in the roadmap.


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

with open('labels.json') as f:
    raw = json.load(f)

df = pd.DataFrame(raw['labels'])
df = df[df['human_relevance'].notna() & df['ai_score'].notna()].copy()
df['human_relevance'] = df['human_relevance'].astype(int)
df['ai_score'] = df['ai_score'].astype(float)
df['human_binary'] = (df['human_relevance'] >= 2).astype(int)
df['ai_binary'] = (df['ai_score'] >= 50).astype(int)
df['agreement'] = (df['human_binary'] == df['ai_binary']).astype(int)

print(f'Loaded: {len(df)} JDs')
print(f'Human distribution: {dict(df["human_relevance"].value_counts().sort_index())}')
print(f'AI score range: {df["ai_score"].min():.0f} - {df["ai_score"].max():.0f}')
df[['jd_id','human_relevance','ai_score','ai_verdict','agreement']].sort_values('human_relevance', ascending=False)

---
## Part 2 — Baseline Overview

Before any metrics: what does the data look like? Are AI scores correlated with human judgements at all?

We expect a positive correlation (r > 0.7) and a roughly monotonic relationship between AI score bins and average human relevance. A flat or negative pattern would indicate the AI is not measuring what we think it is.


In [ ]:
colors_h = {0:'#ef4444', 1:'#f97316', 2:'#f59e0b', 3:'#10b981'}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Career Pilot Track A — Human Labels vs AI Scores', fontsize=13, fontweight='bold')

counts = df['human_relevance'].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color=[colors_h[i] for i in counts.index], edgecolor='white')
axes[0].set_title('Human Relevance Labels')
axes[0].set_xlabel('Score (0-3)')
axes[0].set_ylabel('Count')
for idx, val in counts.items():
    axes[0].text(idx, val+0.1, str(val), ha='center', fontweight='bold')

axes[1].hist(df['ai_score'], bins=10, color='#6366f1', edgecolor='white', linewidth=0.5)
axes[1].axvline(50, color='#f59e0b', linestyle='--', label='threshold 50')
axes[1].set_title('AI Score Distribution')
axes[1].set_xlabel('AI Score (0-100)')
axes[1].set_ylabel('Count')
axes[1].legend()

axes[2].scatter(df['human_relevance'], df['ai_score'],
                c=[colors_h[h] for h in df['human_relevance']], s=80, alpha=0.8, edgecolor='white')
z = np.polyfit(df['human_relevance'], df['ai_score'], 1)
x_line = np.linspace(0, 3, 100)
axes[2].plot(x_line, np.poly1d(z)(x_line), color='#6366f1', linewidth=1.5, alpha=0.7)
axes[2].axhline(50, color='#94a3b8', linestyle='--', linewidth=0.8)
corr = df['human_relevance'].corr(df['ai_score'])
axes[2].set_title(f'Human vs AI  (r = {corr:.2f})')
axes[2].set_xlabel('Human Relevance (0-3)')
axes[2].set_ylabel('AI Score (0-100)')
axes[2].set_xticks([0,1,2,3])

plt.tight_layout()
plt.savefig('results/figures/01_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Pearson r: {corr:.3f}')

---
## Part 2b — Scale Extension: n=200 Dataset

After Track A (n=32 human-labeled), we expanded the dataset to **n=200** by fetching 168 real job descriptions
from Jobicy and The Muse APIs (no-auth). These are scored by the AI but have no human relevance labels.

**Goals of this section:**
1. Confirm the 65-cluster artefact is not a small-sample anomaly — does it replicate at scale?
2. Compare score distributions: human-labeled subset vs AI-only subset.
3. Show verdict distribution across the full pipeline (Apply / Consider / Skip).

Note: all metrics in Parts 3–11 remain anchored to the n=32 labeled set (ground truth required).
The n=200 data is used here as a descriptive distribution check only.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Load full n=200 dataset ──────────────────────────────────────────────────
with open('labels.json') as f:
    raw200 = json.load(f)

all_jds      = raw200['labels']
scored_all   = [j for j in all_jds if j.get('ai_score') is not None]
human_scored = [j for j in scored_all if j.get('human_relevance') is not None]
ai_only      = [j for j in scored_all if j.get('human_relevance') is None]

scores_all   = np.array([j['ai_score'] for j in scored_all])
scores_h     = np.array([j['ai_score'] for j in human_scored])
scores_ao    = np.array([j['ai_score'] for j in ai_only])

print(f'Total JDs:         {len(all_jds)}')
print(f'AI-scored:         {len(scored_all)}')
print(f'  Human-labeled:   {len(human_scored)}')
print(f'  AI-only:         {len(ai_only)}')
print()
print(f'Score stats (all n={len(scored_all)}):')
print(f'  mean   = {scores_all.mean():.1f}')
print(f'  median = {np.median(scores_all):.1f}')
print(f'  SD     = {scores_all.std():.1f}')
print(f'  min    = {scores_all.min():.0f}   max = {scores_all.max():.0f}')
print()
cluster60_70 = (scores_all >= 60) & (scores_all <= 70)
exact65      = scores_all == 65
print(f'65-cluster check (60-70 band): {cluster60_70.sum()} JDs = {100*cluster60_70.mean():.1f}% of scored')
print(f'Exact score = 65:              {exact65.sum()} JDs = {100*exact65.mean():.1f}% of scored')

# ── Verdict distribution ────────────────────────────────────────────────────
from collections import Counter
verdicts = Counter(j.get('ai_verdict', '') for j in scored_all if j.get('ai_verdict'))
print()
print('Verdict distribution:')
for v, c in sorted(verdicts.items(), key=lambda x: -x[1]):
    print(f'  {v:<10s} {c:3d}  ({100*c/len(scored_all):.1f}%)')


In [ ]:
# ── Figure: 3-panel n=200 overview ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Career Pilot — n=200 Dataset Overview (AI-scored)', fontsize=13, fontweight='bold')

# Panel 1 — score distribution: human-labeled vs AI-only
bins = np.arange(0, 105, 5)
axes[0].hist(scores_ao, bins=bins, color='#6366f1', alpha=0.7, label=f'AI-only (n={len(ai_only)})')
axes[0].hist(scores_h,  bins=bins, color='#10b981', alpha=0.9, label=f'Human-labeled (n={len(human_scored)})')
axes[0].axvspan(60, 70, color='#f59e0b', alpha=0.15, label='60–70 band')
axes[0].axvline(65, color='#f59e0b', linestyle='--', linewidth=1.2, alpha=0.8)
axes[0].set_title('Score Distribution\n(human-labeled vs AI-only)')
axes[0].set_xlabel('AI Score (0–100)')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)

# Panel 2 — density around 65 (60-70 zoom)
score_counts = Counter(int(s) for s in scores_all)
x_vals = list(range(55, 76))
y_vals = [score_counts.get(x, 0) for x in x_vals]
bar_colors = ['#f59e0b' if x == 65 else '#6366f1' for x in x_vals]
axes[1].bar(x_vals, y_vals, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[1].set_title(f'65-Cluster Zoom (55–75)\n{exact65.sum()} JDs score exactly 65 ({100*exact65.mean():.1f}%)')
axes[1].set_xlabel('AI Score')
axes[1].set_ylabel('Count')
patch65 = mpatches.Patch(color='#f59e0b', label='Score = 65')
patchOther = mpatches.Patch(color='#6366f1', label='Other scores')
axes[1].legend(handles=[patch65, patchOther], fontsize=8)

# Panel 3 — verdict breakdown
verdict_labels = ['Skip', 'Consider', 'Apply']
verdict_counts = [verdicts.get(v, 0) for v in verdict_labels]
verdict_colors = ['#ef4444', '#f59e0b', '#10b981']
bars = axes[2].bar(verdict_labels, verdict_counts, color=verdict_colors, edgecolor='white')
for bar, cnt in zip(bars, verdict_counts):
    pct = 100 * cnt / len(scored_all)
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{cnt}\n({pct:.0f}%)', ha='center', fontsize=9, fontweight='bold')
axes[2].set_title('Verdict Distribution\n(n=196 AI-scored)')
axes[2].set_ylabel('Count')
axes[2].set_ylim(0, max(verdict_counts) * 1.2)

plt.tight_layout()
plt.savefig('results/figures/02_scale_n200.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/02_scale_n200.png')


### Part 2b — Findings

**The 65-cluster replicates at scale.** At n=196 AI-scored JDs:
- **14.8%** of all scored JDs fall in the 60–70 band
- **12.8%** score *exactly* 65 — a deterministic artefact of the scoring rubric
- The cluster is more visible in AI-only JDs, confirming it is not driven by label composition

**Score distribution:** The AI-only subset (n=164, Jobicy + The Muse jobs) peaks in two bands:
the 10–20 range (clear mismatches) and the 35–45 range (marginal fit). The 65-cluster
forms a third mode — these are borderline-negative JDs where the rubric defaults to 65.

**Verdict breakdown (n=196):**
- Skip: 120 (61%) — majority are genuine mismatches from engineering/marketing/sales categories
- Consider: 58 (30%) — HR and data-adjacent roles
- Apply: 18 (9%) — data science / senior analytics roles

The 9% Apply rate is consistent with a realistic pipeline where most open roles are not strong matches.
The 65-cluster inflates the Consider band slightly — some JDs at 65 should likely be Skip.
This is an open calibration issue documented in **Section 6 (Error Analysis)** and **Section 11 (Threshold Sensitivity)**.


---
## Part 3 — Agreement Accuracy

**Question:** How often does the AI agree with the human decision (apply vs skip)?

Using a binary threshold (human ≥ 2 = relevant, AI ≥ 50 = relevant), we measure:
- **Overall accuracy** — % of JDs where AI and human agree
- **False Positive Rate (FPR)** — % of irrelevant JDs the AI incorrectly marks as relevant (costly: wastes application effort on wrong roles)
- **False Negative Rate (FNR)** — % of relevant JDs the AI misses (costly: missed opportunities)

In a hiring tool, missing a good role (FN) is more costly than reviewing a borderline one (FP). We tolerate some FPR before we accept any FNR.


In [ ]:
total = len(df)
agreed = df['agreement'].sum()
agreement_rate = agreed / total
tp = ((df['human_binary']==1) & (df['ai_binary']==1)).sum()
tn = ((df['human_binary']==0) & (df['ai_binary']==0)).sum()
fp = ((df['human_binary']==0) & (df['ai_binary']==1)).sum()
fn = ((df['human_binary']==1) & (df['ai_binary']==0)).sum()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f'Overall agreement: {agreed}/{total} = {agreement_rate:.0%}')
print(f'True Positives  (both relevant):     {tp}')
print(f'True Negatives  (both not relevant): {tn}')
print(f'False Positives (AI over-scores):    {fp}')
print(f'False Negatives (AI under-scores):   {fn}')
print()
print(f'False Positive Rate: {fpr:.0%}  (target < 15%)  {"✓" if fpr < 0.15 else "✗"}')
print(f'False Negative Rate: {fnr:.0%}  (target < 15%)  {"✓" if fnr < 0.15 else "✗"}')

print('\n--- Disagreements ---')
for _, row in df[df['agreement']==0].iterrows():
    direction = 'AI OVER-scored' if row['ai_binary'] > row['human_binary'] else 'AI UNDER-scored'
    print(f'  {direction}: {row["jd_id"]}')
    print(f'    Human={row["human_relevance"]}/3  AI={row["ai_score"]:.0f}/100  ({row["ai_verdict"]})')
    print(f'    {str(row.get("ai_summary",""))[:110]}')

### 3a — Statistical Uncertainty: Confidence Intervals and Effect Size

Raw accuracy numbers are **point estimates**. With n=32, the sampling uncertainty is substantial — we need confidence intervals to know what the data actually supports, and Cohen's κ to measure agreement corrected for chance.

In [ ]:
import json
import numpy as np
from scipy import stats

with open('labels.json') as f:
    labels = [l for l in json.load(f)['labels']
              if isinstance(l.get('human_relevance'), int) and not l.get('is_synthetic')]

n     = len(labels)
human = np.array([l['human_relevance'] >= 2 for l in labels])
ai    = np.array([l['ai_score'] >= 50        for l in labels])

TP = int(np.sum( human &  ai))
FP = int(np.sum(~human &  ai))
TN = int(np.sum(~human & ~ai))
FN = int(np.sum( human & ~ai))
n_pos, n_neg = TP+FN, FP+TN

acc = (TP+TN)/n
fpr = FP/n_neg
fnr = FN/n_pos if n_pos else 0.0

def wilson_ci(k, n, z=1.96):
    """Wilson score interval — works well even at boundaries (0% or 100%)."""
    if n == 0: return 0.0, 1.0
    p = k/n
    denom  = 1 + z**2/n
    centre = (p + z**2/(2*n)) / denom
    margin = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return max(0.0, centre-margin), min(1.0, centre+margin)

acc_lo, acc_hi = wilson_ci(TP+TN, n)
fpr_lo, fpr_hi = wilson_ci(FP, n_neg)
fnr_lo, fnr_hi = wilson_ci(FN, n_pos)

# Cohen's kappa — adjusts for chance agreement
p_exp = ((TP+FP)/n)*((TP+FN)/n) + ((TN+FN)/n)*((TN+FP)/n)
kappa = (acc - p_exp) / (1 - p_exp) if p_exp < 1 else 0.0
kappa_label = ('Poor (<0.20)' if kappa<0.20 else 'Fair (0.20–0.40)' if kappa<0.40
               else 'Moderate (0.40–0.60)' if kappa<0.60
               else 'Substantial (0.60–0.80)' if kappa<0.80 else 'Almost perfect')

print(f'=== AGREEMENT METRICS WITH UNCERTAINTY  (n={n}) ===')
print(f'  Confusion matrix: TP={TP}  FP={FP}  TN={TN}  FN={FN}')
print(f'  Human pos={n_pos}  neg={n_neg}')
print()
print(f'  {"Metric":<12} {"Estimate":>9}  {"95% Wilson CI":>17}')
print('  ' + '─'*42)
print(f'  {"Accuracy":<12} {acc:>9.1%}  [{acc_lo:.1%}, {acc_hi:.1%}]')
print(f'  {"FPR":<12} {fpr:>9.1%}  [{fpr_lo:.1%}, {fpr_hi:.1%}]')
print(f'  {"FNR":<12} {fnr:>9.1%}  [{fnr_lo:.1%}, {fnr_hi:.1%}]')
print()
print(f"  Cohen's κ = {kappa:.3f}  →  {kappa_label}")
print()
print('  KEY TAKEAWAY:')
print(f'  Accuracy 83% has a 95% CI of [{acc_lo:.0%}, {acc_hi:.0%}] — a ±10pp band.')
print(f'  FPR 44% CI [{fpr_lo:.0%}, {fpr_hi:.0%}] shows it is reliably non-zero,')
print(f'  even accounting for sampling uncertainty with n={n_neg} negatives.')
print(f'  κ={kappa:.3f} = moderate agreement — the gap from "almost perfect" is')
print('  entirely explained by the 4 systematic false positives (FP pattern).')
print('  FNR=0% CI [0%, {:.0%}] confirms the model catches all true positives.'.format(fnr_hi))

---
## Part 4 — Ranking Quality

**Question:** Even if absolute scores aren't perfect, does the AI rank roles in the right order?

A career tool is used like a search engine — the most relevant roles should appear at the top. So ranking quality matters as much as absolute accuracy.

- **NDCG@k** — Normalised Discounted Cumulative Gain: how well do the top-k AI-ranked JDs match human relevance order? (1.0 = perfect)
- **Recall@k** — what fraction of human-relevant roles appear in the top k?
- **MRR** — Mean Reciprocal Rank: how high does the first relevant result appear?


In [ ]:
def dcg(relevances):
    return sum(r / np.log2(i+2) for i, r in enumerate(relevances))

def ndcg_at_k(d, k):
    ranked = d.sort_values('ai_score', ascending=False).head(k)
    ideal  = d.sort_values('human_relevance', ascending=False).head(k)
    idcg = dcg(ideal['human_relevance'].tolist())
    return dcg(ranked['human_relevance'].tolist()) / idcg if idcg > 0 else 0

def recall_at_k(d, k, thr=2):
    good  = d[d['human_relevance'] >= thr]
    top_k = d.sort_values('ai_score', ascending=False).head(k)
    return len(top_k[top_k['human_relevance'] >= thr]) / len(good) if len(good) > 0 else 0

def mrr(d, thr=2):
    for i, row in d.sort_values('ai_score', ascending=False).reset_index(drop=True).iterrows():
        if row['human_relevance'] >= thr:
            return 1 / (i+1)
    return 0

n5, n10 = ndcg_at_k(df,5), ndcg_at_k(df,10)
r5, r10 = recall_at_k(df,5), recall_at_k(df,10)
mrr_val = mrr(df)

print('=== Ranking Accuracy ===')
print(f'NDCG@5:    {n5:.3f}  (target > 0.80)  {"✓" if n5>0.80 else "✗"}')
print(f'NDCG@10:   {n10:.3f}  (target > 0.80)  {"✓" if n10>0.80 else "✗"}')
print(f'Recall@5:  {r5:.3f}  (target > 0.75)  {"✓" if r5>0.75 else "✗"}')
print(f'Recall@10: {r10:.3f}  (target > 0.75)  {"✓" if r10>0.75 else "✗"}')
print(f'MRR:       {mrr_val:.3f}  (target > 0.80)  {"✓" if mrr_val>0.80 else "✗"}')

print('\n--- AI Top 10 vs Human ---')
for i, (_, r) in enumerate(df.sort_values('ai_score', ascending=False).head(10).iterrows(), 1):
    m = '✓' if r['human_relevance'] >= 2 else '✗'
    print(f'  {i:2}. [{r["ai_score"]:3.0f}] H={r["human_relevance"]} {m}  {r["jd_id"][:50]}')

In [ ]:
# Bootstrap confidence intervals for ranking metrics
# Resamples the 24 JDs with replacement to estimate sampling uncertainty.

import json
import numpy as np

np.random.seed(42)
N_BOOT = 5000

with open('labels.json') as f:
    labels = sorted(
        [l for l in json.load(f)['labels']
         if isinstance(l.get('human_relevance'), int) and not l.get('is_synthetic')],
        key=lambda x: -x['ai_score']
    )

n = len(labels)
human = np.array([l['human_relevance'] for l in labels])
ai    = np.array([l['ai_score']         for l in labels])

# Sort by AI score descending (ranking order)
rank_order = np.argsort(-ai)
human_ranked = human[rank_order]

def dcg_at_k(relevances, k):
    rel = relevances[:k]
    return sum(r / np.log2(i+2) for i, r in enumerate(rel))

def ndcg_at_k(relevances, k):
    ideal = sorted(relevances, reverse=True)
    idcg  = dcg_at_k(ideal, k)
    return dcg_at_k(relevances, k) / idcg if idcg > 0 else 0.0

def mrr(relevances, threshold=2):
    for i, r in enumerate(relevances):
        if r >= threshold:
            return 1.0 / (i+1)
    return 0.0

def recall_at_k(relevances, k, threshold=2):
    pos_total = sum(1 for r in relevances if r >= threshold)
    pos_in_k  = sum(1 for r in relevances[:k] if r >= threshold)
    return pos_in_k / pos_total if pos_total > 0 else 0.0

# Point estimates
ndcg5 = ndcg_at_k(human_ranked, 5)
mrr_v = mrr(human_ranked)
rec10 = recall_at_k(human_ranked, 10)

# Bootstrap
boot_ndcg5, boot_mrr, boot_rec10 = [], [], []
indices = np.arange(n)
for _ in range(N_BOOT):
    idx      = np.random.choice(indices, n, replace=True)
    h_boot   = human[idx]
    a_boot   = ai[idx]
    rank_b   = np.argsort(-a_boot)
    h_ranked = h_boot[rank_b]
    boot_ndcg5.append(ndcg_at_k(h_ranked, 5))
    boot_mrr.append(mrr(h_ranked))
    boot_rec10.append(recall_at_k(h_ranked, 10))

def ci(boot): return np.percentile(boot, 2.5), np.percentile(boot, 97.5)

ci_ndcg = ci(boot_ndcg5)
ci_mrr  = ci(boot_mrr)
ci_rec  = ci(boot_rec10)

print('=== RANKING METRICS WITH BOOTSTRAP 95% CI  (B=5000) ===')
print()
print(f'  {"Metric":<15} {"Estimate":>9}  {"95% Bootstrap CI":>18}')
print('  ' + '─'*46)
print(f'  {"NDCG@5":<15} {ndcg5:>9.3f}  [{ci_ndcg[0]:.3f}, {ci_ndcg[1]:.3f}]')
print(f'  {"MRR":<15} {mrr_v:>9.3f}  [{ci_mrr[0]:.3f},  {ci_mrr[1]:.3f}]')
print(f'  {"Recall@10":<15} {rec10:>9.3f}  [{ci_rec[0]:.3f}, {ci_rec[1]:.3f}]')
print()
print('  NDCG@5 = 0.956 with CI [', f'{ci_ndcg[0]:.3f}, {ci_ndcg[1]:.3f}',
      '] — excellent ranking across all resamples.')
print('  MRR CI lower bound > 0.85 — the most relevant role is almost always')
print('  ranked in the top 2 positions, regardless of sample variation.')
print('  Recall@10 CI reflects resampling of a small dataset — 10 of 24 JDs.')
print('  Conclusion: ranking quality is robust; the AI reliably surfaces the')
print('  most relevant roles near the top even when absolute scores are imperfect.')

---
## Part 5 — Calibration

**Question:** Do AI score levels map to real relevance differences?

Calibration measures whether the AI's numerical scale carries meaning. A well-calibrated model should show: AI 75–100 → average human label close to 3, AI 0–25 → average human label close to 0.

We use linear R² (AI score ~ human relevance) as a summary statistic. R² > 0.80 means the AI score explains >80% of variance in human judgement.


In [ ]:
df['score_bin'] = pd.cut(df['ai_score'], bins=[0,25,50,75,100],
                          labels=['0-25','25-50','50-75','75-100'], include_lowest=True)
calib = df.groupby('score_bin', observed=True)['human_relevance'].agg(['mean','count']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Calibration: AI Score vs Human Relevance', fontsize=12, fontweight='bold')

bars = axes[0].bar(range(len(calib)), calib['mean'], color='#6366f1', alpha=0.8, edgecolor='white')
axes[0].set_xticks(range(len(calib)))
axes[0].set_xticklabels(calib['score_bin'])
axes[0].set_xlabel('AI Score Bin')
axes[0].set_ylabel('Avg Human Relevance (0-3)')
axes[0].set_ylim(0, 3.5)
axes[0].axhline(2.0, color='#10b981', linestyle='--', alpha=0.5, label='apply threshold')
axes[0].legend()
for bar, (_, row) in zip(bars, calib.iterrows()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f"{row['mean']:.1f}\nn={int(row['count'])}", ha='center', fontsize=9)

slope, intercept, r_val, pval, _ = stats.linregress(df['ai_score'], df['human_relevance'])
r2 = r_val**2
axes[1].scatter(df['ai_score'], df['human_relevance'],
                c=[{0:'#ef4444',1:'#f97316',2:'#f59e0b',3:'#10b981'}[h] for h in df['human_relevance']],
                s=80, alpha=0.8, edgecolor='white')
x_line = np.linspace(df['ai_score'].min(), df['ai_score'].max(), 100)
axes[1].plot(x_line, slope*x_line+intercept, color='#6366f1', linewidth=2)
axes[1].set_xlabel('AI Score')
axes[1].set_ylabel('Human Relevance')
axes[1].set_title(f'R² = {r2:.3f}  (target > 0.80)  {"✓" if r2>0.80 else "✗"}')

plt.tight_layout()
plt.savefig('results/figures/03_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'R² = {r2:.3f}  slope = {slope:.3f}  p = {pval:.4f}')
print(calib.to_string(index=False))

---
## Part 6 — Error Analysis: The 65 Problem

**Question:** Where exactly does the AI fail, and why?

In the original 24-JD dataset, all 4 false positives scored exactly AI=65. When the dataset was extended to 32 JDs (8 new entries added to strengthen the negative class), **all 4 new false positives also scored exactly 65** — zero exceptions. This is now 8/8 FPs at exactly 65 across two independent batches of JDs.

This is not coincidence. The AI is not failing through miscalibration — it is **defaulting to 65 as a safe "uncertain" score** when it cannot confidently place a JD above or below its decision boundary. The problem is structural, not addressable by prompt tuning.

If this is true, raising the threshold above 65 would eliminate all FPs — but at the risk of also cutting some genuine human=2 roles that score near 65.

Let's look at the AI score distributions for human=1 vs human=2 across all 32 JDs:

*(Note: the threshold sensitivity and prompt experiment sections below use the original 24-JD set.)* are human=1 roles (adjacent domain, candidate would not apply) that received AI score = 65. This is suspicious — it suggests the AI isn't confidently deciding; it's defaulting to a "middle safe" score when uncertain.

If this is true, raising the threshold above 65 would eliminate all FPs — but at the risk of also cutting some genuine human=2 roles that score near 65.

Let's look at the AI score distributions for human=1 vs human=2:


In [ ]:
h1 = df[df['human_relevance']==1]['ai_score']
h2 = df[df['human_relevance']==2]['ai_score']

fig, ax = plt.subplots(figsize=(9, 4))
np.random.seed(42)
ax.scatter(1 + np.random.uniform(-0.05,0.05,len(h1)), h1,
           color='#f97316', s=100, label='Human=1 (would NOT apply)', zorder=3)
ax.scatter(2 + np.random.uniform(-0.05,0.05,len(h2)), h2,
           color='#f59e0b', s=100, label='Human=2 (apply on chance)', zorder=3)
ax.axhline(50, color='#94a3b8', linestyle='--', linewidth=1, label='AI threshold = 50')
ax.axhline(65, color='#ef4444', linestyle=':', linewidth=1.5, label='AI = 65 (all 4 disagreements)')
ax.set_xticks([1,2])
ax.set_xticklabels(['Human=1\n(would not apply)', 'Human=2\n(apply on chance)'])
ax.set_ylabel('AI Score')
ax.set_title('Boundary Problem: Human 1 vs 2  —  AI scores cluster at 65')
ax.legend()
ax.set_xlim(0.7, 2.3)
plt.tight_layout()
plt.savefig('results/figures/04_boundary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Human=1 AI scores: {sorted(h1.tolist())}')
print(f'Human=2 AI scores: {sorted(h2.tolist())}')
print(f'\nHuman=1 mean: {h1.mean():.1f}  std: {h1.std():.1f}')
print(f'Human=2 mean: {h2.mean():.1f}  std: {h2.std():.1f}')
print('\nConclusion: raising AI threshold from 50 → ~68 would fix all 4 disagreements')

---
## Part 7 — Threshold Sensitivity

**Question:** If AI=65 is the problem, what's the optimal threshold?

We sweep the threshold from 45 to 80 and measure FPR, FNR, and accuracy at each cutpoint. The goal: find a threshold that eliminates the 65-cluster FPs without creating new FNs on genuine human=2 roles.

The key question is whether any threshold cleanly separates human=1 from human=2 in the 55–70 AI score range.


In [ ]:
thresholds = list(range(45, 81, 1))
results_thr = []
for thr in thresholds:
    ai_bin = (df['ai_score'] >= thr).astype(int)
    tp = ((df['human_binary']==1) & (ai_bin==1)).sum()
    tn = ((df['human_binary']==0) & (ai_bin==0)).sum()
    fp = ((df['human_binary']==0) & (ai_bin==1)).sum()
    fn = ((df['human_binary']==1) & (ai_bin==0)).sum()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    acc = (tp + tn) / len(df)
    results_thr.append({'thr': thr, 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
                        'fpr': fpr, 'fnr': fnr, 'acc': acc})

import pandas as pd
tdf = pd.DataFrame(results_thr)

# Find optimal: min(FPR + FNR)
tdf['error_sum'] = tdf['fpr'] + tdf['fnr']
best_idx = tdf['error_sum'].idxmin()
best = tdf.loc[best_idx]

print('Threshold sweep (45-80):\n')
print(f'{"Thr":>5} {"FPR":>6} {"FNR":>6} {"FP":>4} {"FN":>4} {"Acc":>6}')
print('-' * 35)
for _, r in tdf.iterrows():
    marker = ' <-- optimal' if r['thr'] == best['thr'] else ''
    marker2 = ' <-- current' if r['thr'] == 50 else ''
    print(f"{r['thr']:>5.0f} {r['fpr']:>6.0%} {r['fnr']:>6.0%} {r['fp']:>4.0f} {r['fn']:>4.0f} {r['acc']:>6.0%}{marker}{marker2}")

# Plot
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Threshold Sensitivity Analysis', fontsize=12, fontweight='bold')

axes[0].plot(tdf['thr'], tdf['fpr']*100, color='#ef4444', linewidth=2, label='False Positive Rate')
axes[0].plot(tdf['thr'], tdf['fnr']*100, color='#f59e0b', linewidth=2, label='False Negative Rate')
axes[0].plot(tdf['thr'], tdf['acc']*100, color='#6366f1', linewidth=2, linestyle='--', label='Accuracy')
axes[0].axvline(50, color='#94a3b8', linestyle=':', linewidth=1.5, label='current threshold=50')
axes[0].axvline(best['thr'], color='#10b981', linestyle='--', linewidth=2, label=f"optimal={best['thr']:.0f}")
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('%')
axes[0].set_title('FPR / FNR / Accuracy vs Threshold')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Show the score distribution with threshold overlaid
axes[1].hist(df[df['human_binary']==1]['ai_score'], bins=15, color='#10b981', alpha=0.6,
             edgecolor='white', label='Human relevant (>=2)')
axes[1].hist(df[df['human_binary']==0]['ai_score'], bins=15, color='#ef4444', alpha=0.6,
             edgecolor='white', label='Human not relevant (<2)')
axes[1].axvline(50, color='#94a3b8', linestyle=':', linewidth=2, label='current=50')
axes[1].axvline(best['thr'], color='#10b981', linestyle='--', linewidth=2, label=f"optimal={best['thr']:.0f}")
axes[1].set_xlabel('AI Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Score distributions: relevant vs not relevant')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/figures/06_threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n=== RECOMMENDATION ===')
print(f'Current threshold:   50  →  FPR=44%  FNR= 0%   Acc=83%  (4 FPs, 0 FNs)')
print(f'Optimal threshold:  {best["thr"]:.0f}  →  FPR={best["fpr"]:.0%}   FNR={best["fnr"]:.0%}   Acc={best["acc"]:.0%}  ({int(best["fp"])} FPs, {int(best["fn"])} FNs)')
print()
print('TRADE-OFF:')
print('  Threshold 50 (current): catches all good roles but flags 4 not-relevant as Consider')
print('  Threshold 66 (optimal): eliminates 4 FPs but misses 2 borderline roles (AI=65 and AI=55)')
print('  Root cause: AI uses 65 as a default score for uncertainty regardless of true relevance.')
print('  No single threshold cleanly separates human=1 from human=2 in the 55-65 AI score range.')
print()
print('DECISION OPTIONS:')
print('  A) Keep threshold=50 — no missed roles, but 4 FPs (current behaviour)')
print('  B) Raise to 66 — eliminates FPs but misses DataAnnotation (AI=65, H=2) and freeplay (AI=55, H=2)')
print('  C) Fix the prompt — instruct AI to not default-score borderline roles at 65')
print('     → Most impactful fix: would spread uncertainty across a wider score range')


---
## Part 8 — Reliability: Test-Retest Consistency

**Question:** If we run the same JD through the AI twice, do we get the same score?

Reliability is a prerequisite for any other metric. If scores vary randomly across runs, all accuracy and calibration numbers are measuring noise rather than model behaviour.

We use **ICC(2,1)** — Intraclass Correlation Coefficient, two-way absolute agreement — the standard reliability metric in psychometric test development. ICC > 0.90 is considered excellent; ICC > 0.80 is acceptable.

**Dataset:** All 24 Track A JDs, each run **3 times** with the P1 prompt:
- **Run O** (original): company name visible in JD text
- **Run M1** (masked): company name → `Company_XX`  
- **Run M2** (masked, repeat): same masked JD, independent API call

This 3-run design comes from the company name bias test (Part 9 extended). It gives us:
- **Pure test-retest ICC** from M1 vs M2 (identical input, two independent calls)
- **Full 3-run ICC** including original (measures total score variance across conditions)
- A dataset 5× larger than the original 5-JD consistency test

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

with open('data/batch-results/company_name_bias.json') as f:
    bias_data = json.load(f)

def icc_2way_absolute(matrix):
    n, k = matrix.shape
    grand = matrix.mean()
    rows  = matrix.mean(axis=1)
    cols  = matrix.mean(axis=0)
    SSr = k * np.sum((rows - grand)**2)
    SSc = n * np.sum((cols - grand)**2)
    SSe = np.sum((matrix - rows[:,None] - cols[None,:] + grand)**2)
    dfr, dfc, dfe = n-1, k-1, (n-1)*(k-1)
    MSr = SSr/dfr
    MSc = SSc/dfc if dfc>0 else 0
    MSe = SSe/dfe if dfe>0 else 1e-9
    icc  = max(0.0, (MSr - MSe) / (MSr + (k-1)*MSe + k*(MSc-MSe)/n))
    F = MSr/MSe
    p = 1 - stats.f.cdf(F, dfr, dfe)
    return float(icc), float(F), dfr, dfe, float(p), float(MSr), float(MSe)

mat_masked = np.array([[r['score_m1'], r['score_m2']] for r in bias_data], dtype=float)
mat_all    = np.array([[r['score_orig'], r['score_m1'], r['score_m2']] for r in bias_data], dtype=float)

icc_m, Fm, df1m, df2m, pm, _, _ = icc_2way_absolute(mat_masked)
icc_a, Fa, df1a, df2a, pa, _, _ = icc_2way_absolute(mat_all)

diffs_m  = [abs(r['score_m1'] - r['score_m2'])   for r in bias_data]
diffs_om = [abs(r['score_orig'] - r['score_mask_mean']) for r in bias_data]

print('=== TEST-RETEST RELIABILITY — 24 JDs × 3 runs ===\n')
print(f'{"JD":<42} {"H":>2}  {"Orig":>4}  {"M1":>4}  {"M2":>4}  {"M diff":>6}  {"SD_all":>6}')
print('─' * 72)
colors_h = {0:'✗', 1:'○', 2:'◑', 3:'●'}
for r in bias_data:
    diff_m = abs(r['score_m1'] - r['score_m2'])
    flag = ' △' if diff_m >= 5 else ''
    print(f'{r["jd_id"][:41]:<42} {r["human_relevance"]:>2}  '
          f'{r["score_orig"]:>4}  {r["score_m1"]:>4}  {r["score_m2"]:>4}  '
          f'{diff_m:>6}  {r["total_sd"]:>6.1f}{flag}')

print()
print('=== ICC SUMMARY ===')
print(f'{"Condition":<30} {"ICC(2,1)":>9}  {"F":>7}  {"df":>8}  {"p":>8}  {"Benchmark"}')
print('─' * 78)
print(f'{"M1 vs M2 (pure test-retest)":<30} {icc_m:>9.3f}  {Fm:>7.2f}  '
      f'{df1m},{df2m:>3}  {pm:>8.4f}  > 0.90 ✓' if icc_m>0.90 else
      f'{"M1 vs M2 (pure test-retest)":<30} {icc_m:>9.3f}  {Fm:>7.2f}  '
      f'{df1m},{df2m:>3}  {pm:>8.4f}')
print(f'{"All 3 runs O+M1+M2":<30} {icc_a:>9.3f}  {Fa:>7.2f}  '
      f'{df1a},{df2a:>3}  {pa:>8.4f}  > 0.90 ✓' if icc_a>0.90 else
      f'{"All 3 runs O+M1+M2":<30} {icc_a:>9.3f}  {Fa:>7.2f}  '
      f'{df1a},{df2a:>3}  {pa:>8.4f}')
print()
print(f'Mean |M1−M2| diff: {np.mean(diffs_m):.1f} pts  |  Max: {max(diffs_m)} pts')
print(f'Mean |Orig−Mask|:  {np.mean(diffs_om):.1f} pts  |  Max: {max(diffs_om):.1f} pts')
print()
print('ICC drop (all 3 vs masked-only):', round(icc_m - icc_a, 4),
      '← variance added by company name condition')

# ── Plots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Reliability: Test-Retest Consistency — 24 JDs × 3 runs', fontsize=13, fontweight='bold')

colors_h_map = {0:'#ef4444', 1:'#f97316', 2:'#f59e0b', 3:'#10b981'}
c = [colors_h_map[r['human_relevance']] for r in bias_data]
m1 = [r['score_m1'] for r in bias_data]
m2 = [r['score_m2'] for r in bias_data]

# Scatter M1 vs M2
axes[0].scatter(m1, m2, color=c, s=80, alpha=0.9, edgecolor='white', zorder=3)
axes[0].plot([0,100],[0,100], '--', color='#94a3b8', lw=1)
axes[0].plot([0,95],[5,100], ':', color='#10b981', lw=0.8, alpha=0.5)
axes[0].plot([5,100],[0,95], ':', color='#10b981', lw=0.8, alpha=0.5)
axes[0].set_xlabel('Run M1'); axes[0].set_ylabel('Run M2')
axes[0].set_title(f'M1 vs M2  (ICC={icc_m:.3f})')
from matplotlib.lines import Line2D
legend_elems = [Line2D([0],[0],marker='o',color='w',markerfacecolor=colors_h_map[h],
                        markersize=8,label=f'H={h}') for h in [0,1,2,3]]
axes[0].legend(handles=legend_elems, fontsize=8)

# Scatter Orig vs M1 (show company name effect)
orig = [r['score_orig'] for r in bias_data]
axes[1].scatter(orig, m1, color=c, s=80, alpha=0.9, edgecolor='white', zorder=3)
axes[1].plot([0,100],[0,100], '--', color='#94a3b8', lw=1)
axes[1].set_xlabel('Run O (original)'); axes[1].set_ylabel('Run M1 (masked)')
axes[1].set_title(f'Original vs Masked  (ICC={icc_a:.3f} all 3)')

# Bar: M1-M2 diff per JD sorted
sorted_data = sorted(zip(diffs_m, [r['jd_id'][:20] for r in bias_data]))
diffs_sorted, labels_sorted = zip(*sorted_data)
bar_colors = ['#10b981' if d<5 else '#f59e0b' if d<10 else '#ef4444' for d in diffs_sorted]
axes[2].barh(range(len(diffs_sorted)), diffs_sorted, color=bar_colors, edgecolor='white')
axes[2].axvline(5, color='#94a3b8', linestyle='--', lw=1, label='5 pts')
axes[2].set_yticks(range(len(labels_sorted)))
axes[2].set_yticklabels(labels_sorted, fontsize=7)
axes[2].set_xlabel('|M1 − M2| score difference')
axes[2].set_title('Masked run variability per JD')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/figures/08_reliability_icc.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('INTERPRETATION:')
print(f'  ICC(M1 vs M2) = {icc_m:.3f} — Excellent reliability on masked JDs.')
print(f'  ICC(all 3)    = {icc_a:.3f} — Near-identical; company name adds minimal variance.')
print(f'  Mean M1–M2 diff = {np.mean(diffs_m):.1f} pts — well within measurement noise.')
print('  Score variance is concentrated in the 50–80 range (borderline cases),')
print('  consistent with findings from the threshold sensitivity analysis.')
print('  Conclusion: model outputs are stable. Observed score variation reflects')
print('  genuine task ambiguity, not random API noise.')

---
## Part 9 — Robustness: CV Paraphrase Test

**Question:** Does rephrasing the CV (without changing content) change the AI's scores?

Candidates typically have one CV, but its wording varies across versions. If the AI is sensitive to phrasing rather than substance, scores would shift when the same experience is described differently.

**Design:** The original 24 Track A JDs run with two CV versions (robustness test predates the n=32 extension):
- `cv_primary.txt` — original CV
- `cv_paraphrased.txt` — same experience, different wording

**Target:** ≤5 pts shift for >80% of roles.


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

with open('data/batch-results/robustness_cv_paraphrase.json') as f:
    rob = json.load(f)

rdf = pd.DataFrame(rob)
rdf['diff'] = rdf['diff'].astype(float)

stable_n = rdf['stable'].sum()
pct = stable_n / len(rdf)
mean_diff = rdf['diff'].mean()
max_diff = rdf['diff'].max()

print(f'Tested:          {len(rdf)}')
print(f'Stable (<=5pts): {stable_n}/{len(rdf)} = {pct:.0%}  (target > 80%)  {"✓" if pct > 0.80 else "✗"}')
print(f'Mean diff:       {mean_diff:.1f} pts')
print(f'Max diff:        {max_diff:.0f} pts')
print()

# Paraphrased CV systematically higher?
rdf['direction'] = rdf['paraphrased_score'] - rdf['primary_score']
pos = (rdf['direction'] > 0).sum()
neg = (rdf['direction'] < 0).sum()
zero = (rdf['direction'] == 0).sum()
print(f'Paraphrased > Primary: {pos} roles')
print(f'Paraphrased < Primary: {neg} roles')
print(f'Equal:                 {zero} roles')
print(f'Mean shift: {rdf["direction"].mean():+.1f} pts (positive = paraphrased scores higher)')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('CV Robustness: Primary vs Paraphrased', fontsize=12, fontweight='bold')

colors = ['#10b981' if s else '#ef4444' for s in rdf['stable']]
rdf_sorted = rdf.sort_values('diff', ascending=False)
axes[0].barh(range(len(rdf_sorted)), rdf_sorted['diff'],
             color=['#10b981' if s else '#ef4444' for s in rdf_sorted['stable']])
axes[0].axvline(5, color='#94a3b8', linestyle='--', linewidth=1, label='threshold = 5 pts')
axes[0].set_xlabel('Score Difference (pts)')
axes[0].set_title('Score Shift per JD')
axes[0].set_yticks(range(len(rdf_sorted)))
axes[0].set_yticklabels([j[:35] for j in rdf_sorted['jd_id']], fontsize=7)
axes[0].legend()

axes[1].scatter(rdf['primary_score'], rdf['paraphrased_score'],
                c=['#10b981' if s else '#ef4444' for s in rdf['stable']],
                s=80, alpha=0.8, edgecolor='white')
mn = min(rdf['primary_score'].min(), rdf['paraphrased_score'].min()) - 5
mx = max(rdf['primary_score'].max(), rdf['paraphrased_score'].max()) + 5
axes[1].plot([mn,mx],[mn,mx], color='#94a3b8', linestyle='--', linewidth=1, label='perfect stability')
axes[1].plot([mn,mx],[mn+5,mx+5], color='#10b981', linestyle=':', linewidth=0.8, alpha=0.5)
axes[1].plot([mn,mx],[mn-5,mx-5], color='#10b981', linestyle=':', linewidth=0.8, alpha=0.5)
axes[1].set_xlabel('Primary CV Score')
axes[1].set_ylabel('Paraphrased CV Score')
axes[1].set_title('Primary vs Paraphrased (green band = ±5 pts)')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/figures/05_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nUnstable JDs:')
for _, r in rdf[~rdf['stable']].sort_values('diff', ascending=False).iterrows():
    h = r.get('human_relevance', '?')
    print(f'  H={h}  {r["primary_score"]:3.0f} → {r["paraphrased_score"]:3.0f}  (+{r["diff"]:.0f} pts)  {r["jd_id"][:50]}')


---
## Part 10 — Discrimination: Synthetic CV Test

**Question:** Does the AI correctly score a wrong-domain candidate low on psychometrics/assessment roles?

This is a discrimination test using the original 24 JDs — we run a synthetic CV (Alex Morgan, Senior UX Researcher at Monzo/Just Eat/Booking.com) against the same 24 JDs. Alex has:

**Has:** BSc Psychology, user research, data synthesis, stakeholder management  
**Does not have:** IRT, CFA, DIF, psychometrics methodology, SQL, Python, ML, people analytics tooling

**Expected:** Alex scores < 40 on all psychometrics/assessment roles. If scores are ≥50, the AI is using surface-level keywords (psychology, research, data) as proxies for domain expertise — a discriminant validity failure.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('data/batch-results/synthetic_cv_scores.json') as f:
    syn = json.load(f)

# Add primary scores from labels.json for comparison
with open('labels.json') as f:
    lab = {l['jd_id']: l for l in json.load(f)['labels']}

for r in syn:
    r['human_relevance'] = lab.get(r['jd_id'], {}).get('human_relevance', r['human_relevance'])

# Classify by role type (for richer breakdown)
hard_psych = ['maki_people_senior_psychometrician',
              'bpostgroup_People_Analytics_Data_Scientist ',
              'Dandelion_Civilization_Psychometric Modeling_Analyst']
soft_eval  = ['codility_assessment_scientist',
              'the_world_bank_consultant_agent_evaluation',
              'skillvue_people_scientist']

colors_h = {0:'#ef4444', 1:'#f97316', 2:'#f59e0b', 3:'#10b981'}

print('=== DISCRIMINATION TEST — Primary vs Synthetic ===\n')
print(f'{"JD":<48} {"H":>3} {"Primary":>8} {"Synth":>6}  {"Gap":>5}  Flag')
print('-' * 85)
for r in sorted(syn, key=lambda x: x['human_relevance'], reverse=True):
    p = r['primary_score'] if isinstance(r['primary_score'], (int,float)) else 0
    s = r['synthetic_score']
    gap = p - s
    flag = ''
    if r['human_relevance'] >= 3 and s >= 50: flag = ' ← FAIL (H=3, synth>=50)'
    elif r['human_relevance'] >= 2 and s >= 50: flag = ' ← WARN (H=2, synth>=50)'
    print(f'{r["jd_id"][:47]:<48} {r["human_relevance"]:>3} {str(p):>8} {s:>6}  {gap:>+5}  {flag}')

print()
all_p = [r['primary_score'] for r in syn if isinstance(r['primary_score'], (int,float))]
all_s = [r['synthetic_score'] for r in syn]
print(f'Primary mean:   {np.mean(all_p):.1f}')
print(f'Synthetic mean: {np.mean(all_s):.1f}  (target < 25)  {"✓" if np.mean(all_s) < 25 else "✗"}')
print(f'Overall gap:    {np.mean(all_p) - np.mean(all_s):+.1f} pts')
print()
print('By human label:')
for h in [3, 2, 1, 0]:
    sub = [r for r in syn if r['human_relevance'] == h]
    pm = np.mean([r['primary_score'] for r in sub if isinstance(r['primary_score'],(int,float))])
    sm = np.mean([r['synthetic_score'] for r in sub])
    print(f'  H={h}:  Primary={pm:.0f}  Synthetic={sm:.0f}  Gap={pm-sm:+.0f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Discrimination Test: Primary (Olga) vs Synthetic (UX Researcher)', fontsize=12, fontweight='bold')

# Scatter: primary vs synthetic, coloured by human label
for r in syn:
    h = r['human_relevance']
    p = r['primary_score'] if isinstance(r['primary_score'],(int,float)) else 0
    axes[0].scatter(p, r['synthetic_score'], color=colors_h.get(h,'#94a3b8'),
                    s=80, alpha=0.85, edgecolor='white')
axes[0].plot([0,100],[0,100], color='#94a3b8', linestyle='--', linewidth=1, label='no discrimination')
axes[0].axhline(50, color='#ef4444', linestyle=':', linewidth=1, alpha=0.6, label='threshold=50')
axes[0].set_xlabel('Primary CV Score (Olga)')
axes[0].set_ylabel('Synthetic CV Score (UX Researcher)')
axes[0].set_title('Primary vs Synthetic')
for h, label in [(0,'H=0'),(1,'H=1'),(2,'H=2'),(3,'H=3')]:
    axes[0].scatter([],[],color=colors_h[h],label=label,s=60)
axes[0].legend(fontsize=8)

# Bar: gap by role (sorted by human label then gap)
syn_sorted = sorted(syn, key=lambda x: (x['human_relevance'], x['primary_score']), reverse=True)
labels_bar = [r['jd_id'][:28] for r in syn_sorted]
gaps = [r['primary_score'] - r['synthetic_score'] if isinstance(r['primary_score'],(int,float)) else 0
        for r in syn_sorted]
bar_colors = [colors_h.get(r['human_relevance'],'#94a3b8') for r in syn_sorted]

axes[1].barh(range(len(syn_sorted)), gaps, color=bar_colors, edgecolor='white', alpha=0.85)
axes[1].axvline(30, color='#94a3b8', linestyle='--', linewidth=1, label='gap=30 pts target')
axes[1].set_yticks(range(len(syn_sorted)))
axes[1].set_yticklabels(labels_bar, fontsize=7)
axes[1].set_xlabel('Primary − Synthetic (pts)')
axes[1].set_title('Discrimination gap per JD')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/figures/10_synthetic_cv.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('=== DIAGNOSIS ===')
print()
print('FAIL cases (H=3, synthetic >= 50):')
for r in syn:
    if r['human_relevance'] >= 3 and r['synthetic_score'] >= 50:
        print(f'  {r["jd_id"]}: synth={r["synthetic_score"]}')
        print(f'    AI reasoning: "{r["synthetic_summary"][:120]}"')

print()
print('ROOT CAUSE:')
print('  Synthetic CV has: BSc Psychology + "research" + "data" + "stakeholder management"')
print('  AI uses these as proxies for psychometrics expertise on soft evaluation roles.')
print('  Specifically: "research methodology" and "Psychology degree" confuse the AI')
print('  on roles whose JD emphasises evaluation design > hard psychometric methods.')
print()
print('DISCRIMINATION IS GOOD FOR:')
print('  - Hard psychometrics (IRT/CFA/DIF explicit): primary 88-95, synthetic 35-40 ✓')
print('  - Technical data science (ML/SQL/Python): primary 70-85, synthetic 20-35 ✓')
print('  - Out-of-domain (design, engineering): both near zero ✓')
print()
print('DISCRIMINATION IS POOR FOR:')
print('  - "Soft" evaluation/research roles (codility, world_bank): gap only 30-37 pts')
print('  - General people analytics without hard tech (Canonical, unicef, Accenture)')
print()
print('IMPLICATION FOR OLGA:')
print('  Practical impact is LOW — primary CV always scores 20-60 pts higher than synthetic.')
print('  The 2 FAIL cases (codility, world_bank) are roles where human judgment should')
print('  override AI — they have a "research generalist" path the AI picks up on.')
print('  Recommended UX fix: show "domain specificity" signal inline in report.')


---
## Part 11 — Full Evaluation Summary (Track A)

All metrics in one table. This is the complete picture of V1 prompt performance across all test dimensions.


In [ ]:
print('=== FULL EVALUATION SUMMARY — Track A ===')
print(f'{"Dimension":<14} {"Metric":<26} {"Value":<10} {"Target":<10} {"Result"}')
print('-' * 70)

rob_stable_pct = stable_n / len(rdf)
rob_mean_diff = mean_diff

all_rows = [
    ('Agreement',   'Overall rate',          f'{agreement_rate:.0%}',  '> 80%',  agreement_rate > 0.80),
    ('Agreement',   'False Positive Rate',   f'{fpr:.0%}',             '< 15%',  fpr < 0.15),
    ('Agreement',   'False Negative Rate',   f'{fnr:.0%}',             '< 15%',  fnr < 0.15),
    ('Ranking',     'NDCG@5',                f'{n5:.3f}',              '> 0.80', n5 > 0.80),
    ('Ranking',     'NDCG@10',               f'{n10:.3f}',             '> 0.80', n10 > 0.80),
    ('Ranking',     'Recall@10',             f'{r10:.3f}',             '> 0.75', r10 > 0.75),
    ('Ranking',     'MRR',                   f'{mrr_val:.3f}',         '> 0.80', mrr_val > 0.80),
    ('Calibration', 'Linear R²',             f'{r2:.3f}',              '> 0.80', r2 > 0.80),
    ('Robustness',  'Stable CVs (<=5 pts)',  f'{rob_stable_pct:.0%}',  '> 80%',  rob_stable_pct > 0.80),
    ('Robustness',  'Mean score shift',      f'{rob_mean_diff:.1f}pt', '< 5 pt', rob_mean_diff < 5),
]
for dim, metric, value, target, passed in all_rows:
    print(f'{dim:<14} {metric:<26} {value:<10} {target:<10} {"✓ PASS" if passed else "✗ FAIL"}')

passed_n = sum(p for *_,p in all_rows)
print(f'\nPassed: {passed_n}/{len(all_rows)}')
print()
print('KEY FINDINGS:')
print('  1. Strong ranking quality — NDCG >0.95, MRR=1.0')
print('  2. FP Rate 44% — AI threshold 50 too low; raise to ~68')
print('  3. Robustness 65% — AI sensitive to CV wording on borderline roles')
print('  4. Paraphrased CV scores +2.7 pts higher on average (n=23, p=0.08, not significant)')
print('  5. Out-of-domain roles: 100% stable (0 diff) — AI confident on extremes')


---
## Part 12 — Prompt Engineering: Can We Fix the FPR?

**Hypothesis:** The FPR problem is caused by the AI's tendency to default-score uncertain cases at 65. If we modify the prompt to discourage 65-clustering, FPR should drop.

We tested three prompt variants:

| Version | Change | Hypothesis |
|---------|--------|------------|
| V1 | Current prompt (no rubric) | Baseline |
| V2 | Anti-clustering instruction + revised rubric | Forces AI away from 65 |
| V3 | Domain specificity gate before scoring | Explicit filter on psychometrics requirement |

All variants were run on the same 24 JDs (original set; prompt experiments predate the n=32 extension) with the same CV.

### V1 vs V2


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

with open('data/batch-results/prompt_comparison_v1_v2.json') as f:
    comp = json.load(f)
cdf = pd.DataFrame(comp)

# Metrics at threshold=50
def metrics(scores, human):
    ai_bin = (scores >= 50).astype(int)
    h_bin  = (human >= 2).astype(int)
    tp = ((h_bin==1) & (ai_bin==1)).sum()
    tn = ((h_bin==0) & (ai_bin==0)).sum()
    fp = ((h_bin==0) & (ai_bin==1)).sum()
    fn = ((h_bin==1) & (ai_bin==0)).sum()
    neg = (h_bin==0).sum()
    pos = (h_bin==1).sum()
    return dict(acc=(tp+tn)/len(scores), fpr=fp/neg if neg else 0,
                fnr=fn/pos if pos else 0, fp=fp, fn=fn, tp=tp, tn=tn)

m1 = metrics(cdf['v1_score'], cdf['human_relevance'])
m2 = metrics(cdf['v2_score'], cdf['human_relevance'])

print('=== PROMPT COMPARISON @ threshold=50 ===')
print(f'{"":22} {"V1 (current)":>14} {"V2 (revised)":>14}')
print('-' * 52)
for key, label in [("acc","Accuracy"),("fpr","FP Rate"),("fnr","FN Rate"),
                    ("fp","FP count"),("fn","FN count")]:
    v1v = f'{m1[key]:.0%}' if isinstance(m1[key], float) else str(m1[key])
    v2v = f'{m2[key]:.0%}' if isinstance(m2[key], float) else str(m2[key])
    print(f'{label:<22} {v1v:>14} {v2v:>14}')

print()
print('SCORE SHIFT — mean by human label:')
for h in [0, 1, 2, 3]:
    sub = cdf[cdf['human_relevance'] == h]
    v1m = sub['v1_score'].mean()
    v2m = sub['v2_score'].mean()
    print(f'  Human={h}: V1 avg={v1m:.1f}  V2 avg={v2m:.1f}  shift={v2m-v1m:+.1f}')

print()
print('KEY FINDING:')
print('  V2 moved ALL uncertain scores down by ~10 pts (mechanical shift).')
print('  4 FPs: 65→55 (still above threshold=50, still false positives).')
print('  2 new FNs: DataAnnotation (65→35), freeplay (55→40) — over-corrected.')
print('  V2 does not improve accuracy — it just shifts the uncertainty cluster.')

# Plot: scatter V1 vs V2 colored by human label
colors_h = {0:'#ef4444', 1:'#f97316', 2:'#f59e0b', 3:'#10b981'}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Prompt V1 vs V2 — Score Comparison', fontsize=12, fontweight='bold')

for _, row in cdf.iterrows():
    h = row['human_relevance']
    axes[0].scatter(row['v1_score'], row['v2_score'],
                    color=colors_h[h], s=80, alpha=0.8, edgecolor='white')
mn, mx = 0, 100
axes[0].plot([mn,mx],[mn,mx], color='#94a3b8', linestyle='--', linewidth=1, label='V1=V2')
axes[0].axhline(50, color='#94a3b8', linestyle=':', linewidth=0.8, alpha=0.5)
axes[0].axvline(50, color='#94a3b8', linestyle=':', linewidth=0.8, alpha=0.5)
axes[0].set_xlabel('V1 Score')
axes[0].set_ylabel('V2 Score')
axes[0].set_title('V1 vs V2 (colour = human label)')
for h, label in [(0,'H=0'),(1,'H=1'),(2,'H=2'),(3,'H=3')]:
    axes[0].scatter([],[],color=colors_h[h],label=label,s=60)
axes[0].legend(fontsize=8)

# Score distribution by human label for V1 and V2 side-by-side
for h in [0, 1, 2, 3]:
    sub = cdf[cdf['human_relevance'] == h]
    axes[1].scatter([f'V1 H={h}']*len(sub), sub['v1_score'],
                    color=colors_h[h], s=60, alpha=0.7, edgecolor='white')
    axes[1].scatter([f'V2 H={h}']*len(sub), sub['v2_score'],
                    color=colors_h[h], s=60, alpha=0.4, edgecolor='black', marker='s')
axes[1].axhline(50, color='#94a3b8', linestyle='--', linewidth=1, label='threshold=50')
axes[1].axhline(65, color='#ef4444', linestyle=':', linewidth=1, alpha=0.5, label='V1 cluster=65')
axes[1].axhline(55, color='#f97316', linestyle=':', linewidth=1, alpha=0.5, label='V2 cluster=55')
axes[1].set_ylabel('AI Score')
axes[1].set_title('Score by label: V1 (circle) vs V2 (square)')
axes[1].legend(fontsize=8)
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig('results/figures/07_prompt_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nROOT CAUSE:')
print('  V2 anti-clustering instruction works — AI stopped using 65 as default.')
print('  But it now defaults to 55 instead. Same structural problem, shifted by 10.')
print('  The AI cannot distinguish human=1 from human=2 using rubric bands alone.')
print('  Reason: these roles genuinely share data/analytics keywords with the CV.')
print()
print('NEEDED: a domain-specificity filter — the real separator is:')
print('  human=1 = adjacent HR analytics (no psychometrics core requirement)')
print('  human=2 = data science with quantitative/measurement elements')
print('V3 prompt should add: explicit domain gate on psychometrics/measurement as criterion.')


---
### V1 vs V2 vs V3 — Three-Way Comparison and Conclusion


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('data/batch-results/prompt_comparison_v1_v2_v3.json') as f:
    v3 = json.load(f)
vdf = pd.DataFrame(v3)

def metrics_at50(score_col, human_col):
    ai = (score_col >= 50).astype(int)
    h  = (human_col >= 2).astype(int)
    fp = ((h==0) & (ai==1)).sum()
    fn = ((h==1) & (ai==0)).sum()
    tp = ((h==1) & (ai==1)).sum()
    tn = ((h==0) & (ai==0)).sum()
    neg = (h==0).sum()
    pos = (h==1).sum()
    return dict(fp=fp, fn=fn, tp=tp, tn=tn,
                fpr=fp/neg if neg else 0, fnr=fn/pos if pos else 0,
                acc=(tp+tn)/len(score_col))

m1 = metrics_at50(vdf['v1_score'], vdf['human_relevance'])
m2 = metrics_at50(vdf['v2_score'], vdf['human_relevance'])
m3 = metrics_at50(vdf['v3_score'], vdf['human_relevance'])

print('=== THREE-WAY PROMPT COMPARISON @ threshold=50 ===')
print(f'{"":22} {"V1 current":>12} {"V2 anti-cluster":>15} {"V3 domain gate":>15}')
print('-' * 68)
for key, label in [("acc","Accuracy"),("fpr","FP Rate"),("fnr","FN Rate"),("fp","FP count"),("fn","FN count")]:
    fmt = lambda v: f'{v:.0%}' if isinstance(v, float) else str(v)
    print(f'{label:<22} {fmt(m1[key]):>12} {fmt(m2[key]):>15} {fmt(m3[key]):>15}')

print()

# Mean score by human label
print('Mean score by human label:')
for h in [0, 1, 2, 3]:
    sub = vdf[vdf['human_relevance'] == h]
    v1m = sub['v1_score'].mean()
    v2m = sub['v2_score'].mean()
    v3m = sub['v3_score'].mean()
    print(f'  H={h}: V1={v1m:.0f}  V2={v2m:.0f}  V3={v3m:.0f}')

# Plot: score distributions per version
colors_h = {0:'#ef4444', 1:'#f97316', 2:'#f59e0b', 3:'#10b981'}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Prompt V1 / V2 / V3 — Score Distributions by Human Label', fontsize=12, fontweight='bold')

for ax, col, title in zip(axes, ['v1_score','v2_score','v3_score'],
                            ['V1 (current)','V2 (anti-cluster)','V3 (domain gate)']):
    for h in [0, 1, 2, 3]:
        sub = vdf[vdf['human_relevance'] == h]
        ax.scatter([h]*len(sub), sub[col], color=colors_h[h], s=70, alpha=0.8, edgecolor='white')
    ax.axhline(50, color='#94a3b8', linestyle='--', linewidth=1, label='threshold=50')
    ax.set_xticks([0,1,2,3])
    ax.set_xticklabels(['H=0','H=1','H=2','H=3'])
    ax.set_ylabel('AI Score')
    ax.set_title(title)
    ax.set_ylim(-5, 105)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/figures/08_v3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== CONCLUSION ===')
print()
print('V1 (current prompt) is the best-performing version:')
print(f'  FPR = {m1["fpr"]:.0%}, FNR = {m1["fnr"]:.0%}, Accuracy = {m1["acc"]:.0%}')
print()
print('WHY PROMPT ENGINEERING CANNOT FIX THE FPR:')
print('  The 4 FP roles (human=1) and borderline human=2 roles look identical')
print('  to the AI from a CV-to-JD matching standpoint — both have data/analytics overlap.')
print('  The separating factor is Olgas application strategy:')
print('  she skips HRIS ops / product analytics / HR generalist roles regardless of data skills.')
print('  This is a strategic career decision, not detectable from text matching.')
print()
print('ALL THREE PROMPTS CLUSTER UNCERTAIN ROLES AT THE SAME SCORE:')
print('  V1: clusters at 65  ("Consider" zone)')
print('  V2: clusters at 55  (same zone, shifted by -10)')
print('  V3: domain gate too strict — misclassifies people-analytics roles (bpostgroup H=3 → 58)')
print()
print('DECISION: KEEP V1 PROMPT. Fix the UX instead:')
print('  Show domain gate reasoning inline: e.g., "No explicit psychometrics requirement"')
print('  Let the user decide on borderline cases with better context, not the AI.')


---
### New Experiments: P0 / P1 / P2

The V1→V2→V3 attempts above used the same approach: adjust rubric wording or add instructions. None fixed the FPR.

We ran three new controlled experiments on the original 24 JDs (prompt experiments predate the n=32 extension):

| Version | What changed | Key question |
|---------|-------------|--------------|
| **P0** | Minimal prompt — 4 score bands only, no rubric, no candidate context | True floor: how much does structure help at all? |
| **P1** | Option B rubric (Domain 35%, Technical 25%, Seniority 20%, Education 15%, Strategic 5%) + candidate hidden context | Does the current app prompt improve over V1 baseline? |
| **P2** | Same as P1 + chain-of-thought: `REASONING:` written before `SCORE:` | Does explicit reasoning reduce clustering and improve calibration? |

Scripts: `evaluation/prompt_p0_simple.py`, `evaluation/prompt_p1_revised_rubric.py`, `evaluation/prompt_p2_cot.py`


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load V1 baseline from labels.json
with open('labels.json') as f:
    raw = json.load(f)
v1_map = {l['jd_id']: l for l in raw['labels']
          if l.get('human_relevance') is not None and l.get('ai_score') is not None}

# Load P0, P1, P2 results
p0 = {r['jd_id']: r for r in json.load(open('data/batch-results/prompt_p0.json'))}
p1 = {r['jd_id']: r for r in json.load(open('data/batch-results/prompt_p1.json'))}
p2 = {r['jd_id']: r for r in json.load(open('data/batch-results/prompt_p2.json'))}

# Build unified dataframe
rows = []
for jd_id, v in v1_map.items():
    rows.append({
        'jd_id': jd_id,
        'human': v['human_relevance'],
        'v1':    v['ai_score'],
        'p0':    p0.get(jd_id, {}).get('p0_score'),
        'p1':    p1.get(jd_id, {}).get('p1_score'),
        'p2':    p2.get(jd_id, {}).get('p2_score'),
    })
pdf = pd.DataFrame(rows).dropna()
pdf['h_bin'] = (pdf['human'] >= 2).astype(int)

def metrics_col(col):
    ai_bin = (pdf[col] >= 50).astype(int)
    tp = ((pdf['h_bin']==1) & (ai_bin==1)).sum()
    tn = ((pdf['h_bin']==0) & (ai_bin==0)).sum()
    fp = ((pdf['h_bin']==0) & (ai_bin==1)).sum()
    fn = ((pdf['h_bin']==1) & (ai_bin==0)).sum()
    neg = (pdf['h_bin']==0).sum(); pos = (pdf['h_bin']==1).sum()
    return dict(acc=(tp+tn)/len(pdf), fpr=fp/neg if neg else 0,
                fnr=fn/pos if pos else 0, fp=int(fp), fn=int(fn),
                r=pdf['human'].corr(pdf[col]))

versions = {'V1 (baseline)':'v1', 'P0 (minimal)':'p0',
            'P1 (rubric+context)':'p1', 'P2 (CoT)':'p2'}

print(f'{"":22}', end='')
for label in versions: print(f' {label:>20}', end='')
print()
print('-' * (22 + 21*len(versions)))

for key, label in [('acc','Accuracy'),('fpr','FP Rate'),('fnr','FN Rate'),
                   ('fp','FP count'),('fn','FN count'),('r','r vs human')]:
    fmt = lambda v, k=key: (f'{v:.0%}' if k in ('acc','fpr','fnr')
                             else (f'{v:.3f}' if k=='r' else str(int(v))))
    print(f'{label:<22}', end='')
    for col in versions.values():
        m = metrics_col(col)
        print(f' {fmt(m[key]):>20}', end='')
    print()

print()
print('Mean score by human label:')
for h in [0,1,2,3]:
    sub = pdf[pdf['human']==h]
    line = f'  H={h}:'
    for name, col in versions.items():
        line += f'  {col.upper()}={sub[col].mean():.0f}'
    print(line)

print()
print('65-clustering (JDs scored exactly 65):')
for name, col in versions.items():
    at65 = int((pdf[col] == 65).sum())
    print(f'  {col.upper()}: {at65} JDs scored exactly 65')


In [ ]:
# ── Score distributions by human label ───────────────────────────────────────
colors_h = {0:'#ef4444', 1:'#f97316', 2:'#f59e0b', 3:'#10b981'}
version_cols = [('V1 baseline','v1'), ('P0 minimal','p0'),
                ('P1 rubric+context','p1'), ('P2 CoT','p2')]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Prompt Experiments P0/P1/P2 — Score Distributions by Human Label',
             fontsize=13, fontweight='bold')

for ax, (title, col) in zip(axes, version_cols):
    for h in [0,1,2,3]:
        sub = pdf[pdf['human']==h]
        ax.scatter([h]*len(sub), sub[col], color=colors_h[h],
                   s=70, alpha=0.8, edgecolor='white')
    ax.axhline(50, color='#94a3b8', linestyle='--', linewidth=1, label='threshold=50')
    ax.axhline(65, color='#ef4444', linestyle=':', linewidth=1, alpha=0.6, label='65 cluster')
    ax.set_xticks([0,1,2,3])
    ax.set_xticklabels(['H=0','H=1','H=2','H=3'])
    ax.set_ylabel('AI Score')
    ax.set_title(title)
    ax.set_ylim(-5, 105)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('results/figures/11_p012_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Accuracy / FPR / FNR bar chart ───────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 4))
labels = list(versions.keys())
x = np.arange(len(labels)); w = 0.25
acc_vals = [metrics_col(c)['acc'] for c in versions.values()]
fpr_vals = [metrics_col(c)['fpr'] for c in versions.values()]
fnr_vals = [metrics_col(c)['fnr'] for c in versions.values()]
ax2.bar(x - w, [v*100 for v in acc_vals], w, color='#6366f1', alpha=0.8, label='Accuracy')
ax2.bar(x,     [v*100 for v in fpr_vals], w, color='#ef4444', alpha=0.8, label='FP Rate')
ax2.bar(x + w, [v*100 for v in fnr_vals], w, color='#f59e0b', alpha=0.8, label='FN Rate')
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=10)
ax2.set_ylabel('%'); ax2.set_title('Accuracy / FPR / FNR by Prompt Version')
ax2.legend(); ax2.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('results/figures/12_p012_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()


---
### Findings

**FPR is stuck at 44% across all versions.** The same 4 JDs (human=1) are false positives in V1, P0, P1, and P2. No prompt change fixes this.

| Version | Accuracy | FPR | FNR | r | 65-cluster |
|---------|----------|-----|-----|---|------------|
| V1 baseline | 83% | 44% | 0% | 0.873 | 5 JDs |
| P0 minimal | 79% | 44% | 7% | 0.857 | 7 JDs |
| P1 rubric+context | **83%** | 44% | **0%** | 0.859 | **0 JDs** |
| P2 CoT | 79% | 44% | 7% | **0.882** | **0 JDs** |

**65-clustering:** P1 and P2 both eliminate it (0 JDs at exactly 65). The rubric forces the AI to commit to a band instead of defaulting to the middle value. But eliminating clustering doesn't reduce FPR — uncertain H=1 roles just get scored at 68–72 instead of 65, still above threshold=50.

**Why CoT (P2) doesn't fix FPR:** The AI reasoning on the 4 FP roles shows it correctly identifies transferable skills but over-generalises. For Arrive (H=1): *"candidate's capabilities far exceed typical HR reporting."* The AI is right that skills transfer — but wrong that Olga would apply. The decision is strategic (she skips HR generalist roles regardless), not detectable from text matching alone.

**P0 and P2 create a new FN:** freeplay (H=2) drops to ~40, below the threshold. This is a regression vs V1.

**Decision: P1 is the best prompt for the current app.** Same accuracy as V1 on the original 24 JDs (83%), same FNR (0%), eliminates 65-clustering, and includes the full rubric + candidate context. The FPR stays at 44% — this requires a UX fix (show reasoning inline on borderline roles), not a prompt fix.


### 12b — Prompt Comparison: Statistical Tests and Power Analysis

The prompt comparison shows identical FPR across all versions (on the original 24 JDs). But are the small differences in FNR (e.g., P0 and P2 each produce 1 FN) statistically distinguishable from noise? McNemar's test answers this for paired binary outcomes.

Power analysis answers the harder question: **with the original n=24 negatives (n_neg=9), what differences could we reliably detect?** This contextualises what null results mean.

In [ ]:
import json
import numpy as np
from scipy.stats import chi2, norm

# Load prompt results
with open('data/batch-results/prompt_p0.json') as f:
    p0_map = {r['jd_id']: r for r in json.load(f)}
with open('data/batch-results/prompt_p1.json') as f:
    p1_map = {r['jd_id']: r for r in json.load(f)}
with open('data/batch-results/prompt_p2.json') as f:
    p2_map = {r['jd_id']: r for r in json.load(f)}
with open('labels.json') as f:
    labels = {l['jd_id']: l for l in json.load(f)['labels']
              if isinstance(l.get('human_relevance'), int) and not l.get('is_synthetic')}

# Build correctness vectors for each prompt on shared JDs
jd_ids = sorted(set(p0_map) & set(p1_map) & set(p2_map) & set(labels))
n = len(jd_ids)

def correct(score, human, threshold=50, pos_threshold=2):
    pred_pos = score >= threshold
    true_pos = human >= pos_threshold
    return pred_pos == true_pos

v1  = np.array([correct(labels[j]['ai_score'],       labels[j]['human_relevance']) for j in jd_ids])
p0v = np.array([correct(p0_map[j]['p0_score'],       labels[j]['human_relevance']) for j in jd_ids])
p1v = np.array([correct(p1_map[j]['p1_score'],       labels[j]['human_relevance']) for j in jd_ids])
p2v = np.array([correct(p2_map[j]['p2_score'],       labels[j]['human_relevance']) for j in jd_ids])

def mcnemar(a, b):
    """McNemar's test: a=correct in condition A, b=correct in condition B.
    Cells: b=True&a=False (c), b=False&a=True (d). Chi2 = (|c-d|-1)^2/(c+d)."""
    c = int(np.sum(~a &  b))   # B correct, A wrong
    d = int(np.sum( a & ~b))   # A correct, B wrong
    if c+d == 0: return 1.0, c, d
    chi2_val = (abs(c-d)-1)**2 / (c+d)   # continuity correction
    p = 1 - chi2.cdf(chi2_val, df=1)
    return float(p), c, d

pairs = [
    ('V1 vs P0', v1, p0v),
    ('V1 vs P1', v1, p1v),
    ('V1 vs P2', v1, p2v),
    ('P1 vs P0', p1v, p0v),
    ('P1 vs P2', p1v, p2v),
]

print(f'=== McNEMAR TEST — PAIRED PROMPT COMPARISON  (n={n}) ===')
print()
print(f'  {"Comparison":<15} {"Acc A":>6}  {"Acc B":>6}  {"c":>3}  {"d":>3}  {"p-value":>9}  {"Sig?"}')
print('  ' + '─'*58)

results_mcn = {}
for name, a, b in pairs:
    acc_a = a.mean(); acc_b = b.mean()
    p, c, d = mcnemar(a, b)
    sig = '✓ p<.05' if p < 0.05 else 'n.s.'
    results_mcn[name] = p
    print(f'  {name:<15} {acc_a:>6.1%}  {acc_b:>6.1%}  {c:>3}  {d:>3}  {p:>9.4f}  {sig}')

print()
print('  c = cases where B correct but A wrong')
print('  d = cases where A correct but B wrong')
print()
print('  All McNemar p-values > 0.05 → no prompt version produces statistically')
print('  different error patterns. The FNs in P0 and P2 are not significant.')

# ── Power analysis ──────────────────────────────────────────────────────────
print()
print('=== POWER ANALYSIS: what can n=24 reliably detect? ===')
print()
print('  Scenario: detecting a reduction in FPR from 44% to X%.')
print('  FPR is measured on n_neg=9 negative JDs.')
print()
print(f'  {"Target FPR":>12}  {"FPR reduction":>14}  {"Required n_neg":>15}  {"Detectable at n=9?"}')
print('  ' + '─'*66)

alpha = 0.05; power = 0.80
p0_fpr = 4/9   # baseline FPR on 9 negatives

for target in [0.33, 0.22, 0.11, 0.00]:
    p1_fpr = target
    if p0_fpr == p1_fpr:
        continue
    # Required n for two-proportion z-test
    z_alpha = norm.ppf(1-alpha/2)
    z_beta  = norm.ppf(power)
    p_bar   = (p0_fpr + p1_fpr) / 2
    if p_bar*(1-p_bar) <= 0 or p0_fpr*(1-p0_fpr) + p1_fpr*(1-p1_fpr) <= 0:
        n_req = 'N/A'
    else:
        se = np.sqrt(2*p_bar*(1-p_bar))
        se2 = np.sqrt(p0_fpr*(1-p0_fpr) + p1_fpr*(1-p1_fpr))
        n_req = int(np.ceil(((z_alpha*se + z_beta*se2)/(p0_fpr-p1_fpr))**2))
    detectable = '✓ YES' if isinstance(n_req, int) and n_req <= 9 else f'✗ Need n≥{n_req}'
    reduction  = f'{p0_fpr-p1_fpr:.0%} → {p1_fpr:.0%}'
    print(f'  {target:>12.0%}  {reduction:>14}  {str(n_req):>15}  {detectable}')

print()
print('  CONCLUSION: With only 9 negative JDs, the study is underpowered to detect')
print('  any plausible FPR reduction. Even eliminating ALL false positives (FPR=0)')
print('  would not reach significance. This is the fundamental constraint of Track A.')
print('  Track B (n≥68 historical jobs) provides the statistical power needed')
print('  to meaningfully test FPR predictors.')

---
## Part 13 — Phase 1: Structured Matching Layer

**What we built:** A deterministic pre-screening layer that scores CV–JD fit before the LLM call, using four components:

| Component | V1 Weight | V2 Weight | Method |
|-----------|-----------|-----------|--------|
| Skills match | 40% | 35% | Keyword vocabulary lookup |
| Seniority match | 35% | 28% | Level patterns (Senior/Lead/Head) |
| Location match | 25% | 17% | Remote/on-site detection |
| Education match | — | 20% | Degree level comparison |

**Original goals:**
1. Hard NOs filter: skip LLM call entirely for roles with dealbreaker attributes (saves cost)
2. Structured score for UI: show a "deterministic baseline" alongside the AI score
3. Context injection: pass structured results into LLM prompt as additional signal

---

### What we found

**Hard NOs filter** — the only part that works as intended. If a role requires 5 days on-site and the candidate is remote-only, skip it without an LLM call. Binary, reliable, zero cost.

**Structured score** — does not correlate with human judgement:
- V1 structured score: r = −0.33, R² = 0.11
- V2 structured score: r = −0.24, R² = 0.06
- V1 LLM score: r = 0.87, R² = 0.76

The root cause: the skills vocabulary is engineering-focused (Python, SQL, React, Docker...). For psychometrics/people analytics JDs, key terms like IRT, CFA, DIF, Rasch, item calibration are not in the dictionary. Result: the skills score defaults to 100% (no required skills detected) — inflating structured scores for irrelevant roles.

**Context injection** — LLM ignores it:

We tested whether injecting the structured pre-screening results into the LLM prompt actually changed AI scores. The sensitivity test (5 JDs × 2 runs: V1 context vs V2 context) showed a **mean shift of 2.2 pts** — below the 3 pt "decorative" threshold.

The structured context block is text the LLM reads but doesn't act on. It's not influencing the AI's reasoning.

---

### Decision

| Component | Decision | Reason |
|-----------|----------|--------|
| Hard NOs filter | **Keep** | Binary, reliable, reduces cost on dealbreaker roles |
| Structured score in UI | **Remove** | Doesn't correlate with human judgement; misleading |
| Context injection into prompt | **Remove** | LLM ignores it (mean shift < 3 pts) |

**Important caveat:** This conclusion is specific to this candidate domain (psychometrics / people data science). The engineering skills vocabulary might work well for software engineering roles. If the vocabulary is extended or replaced with LLM-based skill extraction, the structured layer is worth revisiting.


---
## Part 14 — What's Next: Track B (Predictive Validity)

All Track A experiments complete (Parts 1–13 + Parts 15–16 below). One open question remains: **predictive validity** — does the AI score predict real hiring outcomes?

| Track | Question | Status |
|-------|----------|--------|
| A — Construct validity | Does AI agree with expert human labels? | ✅ Complete (Parts 1–13) |
| C — Multi-model | Is FPR Gemini-specific or task-fundamental? | ✅ Complete (Part 15) |
| D — Annotation audit | Are human labels correct? | ✅ Complete (Part 16) |
| **B — Predictive validity** | **Does AI score predict interview vs rejection?** | ⏳ Post-publication |

**Why Track B runs after publication:** Data is still accumulating (~7 positive outcomes, need ≥10–15 for AUC-ROC). Scripts ready: `evaluation/track_b_prepare.py` + `evaluation/track_b_analyze.py`.

---
## Part 15 — Multi-Model Comparison (Track C)

**Research question:** Is FPR=50% a Gemini-specific calibration failure, or a fundamental limit of the task?

If all models produce the same false positives → the borderline JDs are genuinely ambiguous and no model can reliably distinguish them. If only Gemini fails → a different model could reduce FPR.

**Models tested** (all on 32 masked JDs, P1 prompt or simplified equivalent):
| Model | Type | Cost |
|-------|------|------|
| Gemini V1 | Baseline (from labels.json) | Already computed |
| GPT-4o | Cloud, OpenAI | ~$0.10 |
| deepseek-r1:8b | Local, Ollama | Free |

All JDs masked (Company_XX) to prevent reputation bias.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

with open('data/batch-results/multi_model_comparison.json') as f:
    mm_data = json.load(f)

# Detect models present
model_keys = sorted({k for r in mm_data for k in r.get('scores', {})})
complete = [r for r in mm_data
            if all(r.get('scores', {}).get(m, {}).get('score') is not None for m in model_keys)]

n = len(complete)
n_neg = sum(1 for r in complete if r['human_relevance'] <= 1)
n_pos = sum(1 for r in complete if r['human_relevance'] >= 2)

all_labels = ['Gemini V1'] + model_keys
score_arrays = {'Gemini V1': np.array([r['v1_score'] for r in complete], dtype=float)}
for m in model_keys:
    score_arrays[m] = np.array([r['scores'][m]['score'] for r in complete], dtype=float)

# ── Metrics per model ────────────────────────────────────────────────────────
rows = []
for label in all_labels:
    s_arr = score_arrays[label]
    tp=fp=tn=fn=fps65=0
    for r, s in zip(complete, s_arr):
        h = r['human_relevance']
        if h>=2 and s>=50:   tp+=1
        elif h<=1 and s>=50: fp+=1; fps65 += (1 if s==65 else 0)
        elif h<=1 and s<50:  tn+=1
        elif h>=2 and s<50:  fn+=1
    rows.append({'model': label, 'acc': (tp+tn)/n, 'fpr': fp/n_neg,
                 'fnr': fn/n_pos, 'fp': fp, 'fps65': fps65, 'mean': s_arr.mean()})

print(f'Multi-model comparison (n={n}, pos={n_pos}, neg={n_neg})')
print(f'{"Model":<20} {"Acc":>6}  {"FPR":>6}  {"FNR":>6}  {"FPs@65":>8}  {"Mean score":>10}')
print('─' * 60)
for r in rows:
    tag = '  ← baseline' if r['model']=='Gemini V1' else ''
    print(f'{r["model"]:<20} {r["acc"]:>6.1%}  {r["fpr"]:>6.1%}  {r["fnr"]:>6.1%}  '
          f'{r["fps65"]}/{r["fp"]:>2} FPs  {r["mean"]:>10.1f}{tag}')

# ── ICC ───────────────────────────────────────────────────────────────────────
def icc_2way(matrix):
    n, k = matrix.shape
    grand = matrix.mean(); rows_m = matrix.mean(axis=1); cols_m = matrix.mean(axis=0)
    SSr = k*np.sum((rows_m-grand)**2); SSc = n*np.sum((cols_m-grand)**2)
    SSe = np.sum((matrix - rows_m[:,None] - cols_m[None,:] + grand)**2)
    MSr=SSr/(n-1); MSc=SSc/(k-1) if k>1 else 0; MSe=SSe/((n-1)*(k-1)) if k>1 else 1e-9
    return float(max(0, (MSr-MSe)/(MSr+(k-1)*MSe+k*(MSc-MSe)/n)))

mat_all = np.column_stack([score_arrays[l] for l in all_labels])
icc_val = icc_2way(mat_all)
print(f'\nICC (all models): {icc_val:.3f}')
print('Pairwise Pearson r:')
for i, l1 in enumerate(all_labels):
    for l2 in all_labels[i+1:]:
        r_val = np.corrcoef(score_arrays[l1], score_arrays[l2])[0,1]
        print(f'  {l1} vs {l2}: r={r_val:.3f}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Track C — Multi-model Comparison', fontsize=13, fontweight='bold')

# Left: Accuracy / FPR / FNR bar chart
x = np.arange(len(rows))
w = 0.25
ax = axes[0]
colors = ['#10b981', '#ef4444', '#f59e0b']
labels_bar = ['Accuracy', 'FPR', 'FNR']
vals = [[r['acc'] for r in rows], [r['fpr'] for r in rows], [r['fnr'] for r in rows]]
for i, (label, vals_i, c) in enumerate(zip(labels_bar, vals, colors)):
    ax.bar(x + i*w - w, vals_i, w, label=label, color=c, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([r['model'] for r in rows], rotation=15, ha='right')
ax.set_ylabel('Rate')
ax.set_ylim(0, 1.0)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.legend()
ax.set_title('Accuracy, FPR, FNR by Model')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Right: Score scatter — Gemini vs GPT-4o
ax2 = axes[1]
if 'gpt-4o' in score_arrays:
    g = score_arrays['Gemini V1']
    gpt = score_arrays['gpt-4o']
    h_vals = np.array([r['human_relevance'] for r in complete])
    c_map = {0:'#ef4444', 1:'#f97316', 2:'#f59e0b', 3:'#10b981'}
    for hv in [0,1,2,3]:
        mask = h_vals == hv
        ax2.scatter(g[mask], gpt[mask], c=c_map[hv], s=50, alpha=0.8, label=f'H={hv}', zorder=3)
    lims = [0, 100]
    ax2.plot(lims, lims, 'k--', alpha=0.3, linewidth=1)
    ax2.axvline(50, color='gray', linestyle=':', alpha=0.5)
    ax2.axhline(50, color='gray', linestyle=':', alpha=0.5)
    r_val = np.corrcoef(g, gpt)[0,1]
    ax2.set_xlabel('Gemini V1 score'); ax2.set_ylabel('GPT-4o score')
    ax2.set_title(f'Gemini vs GPT-4o  (r={r_val:.3f})')
    ax2.legend(title='Human label', fontsize=8)
    ax2.set_xlim(-5, 105); ax2.set_ylim(-5, 105)

plt.tight_layout()
plt.savefig('results/figures/multi_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── FP overlap ────────────────────────────────────────────────────────────────
neg_rows = [r for r in complete if r['human_relevance'] <= 1]
print(f'\nFP cases (H≤1) — which models flag them as FP (score≥50)?')
print(f'{"JD":<45} {"H":>2}  {"Gemini":>7}', end='')
for m in model_keys: print(f'  {m[:10]:>10}', end='')
print()
print('─'*70)
for r in sorted(neg_rows, key=lambda x: x['v1_score'], reverse=True):
    v1s = r['v1_score']
    print(f'{r["jd_id"][:44]:<45} {r["human_relevance"]:>2}  {v1s:>6}{"*" if v1s>=50 else " "}', end='')
    for m in model_keys:
        ms = r['scores'][m]['score']
        print(f'  {ms:>9}{"*" if ms>=50 else " "}', end='')
    print()
print('* = FP (score≥50)')

---
### Part 15 — Findings

**FPR varies across models** (range = 31pp), confirming the problem is **partially model-specific**, not purely task-fundamental:

| Model | FPR | FPs@65 | Interpretation |
|-------|-----|--------|----------------|
| Gemini V1 | **50%** | 8/8 at exactly 65 | Lowest FPR; 65-clustering is model-specific behaviour |
| GPT-4o | 63% | 0/10 at 65 | Higher FPR; no 65-clustering — uses varied scores |
| deepseek-r1:8b | 81% | 4/13 | Highest FPR; most permissive on borderline cases |

**Gemini is the best-calibrated model on this task** — despite the 65-default behaviour, it produces fewer false positives than both alternatives. The 65-clustering is a Gemini-specific artefact, but the underlying task difficulty affects all models.

**ICC(all models) = 0.784** — models agree moderately on scores overall. Gemini–GPT-4o correlation (r=0.917) is much higher than Gemini–deepseek (r=0.771), suggesting GPT-4o and Gemini share similar evaluation heuristics for this domain.

---
## Part 16 — LLM Annotation Audit (Track D)

**Research question:** Are the human labels correct? Do independent LLM judges agree with the 0–3 relevance ratings — especially on the 8 false-positive cases?

**Key question for the FP cases:** AI scored these 8 JDs ≥50 but human labeled them 0–1.
- If LLM judges say H≤1 → human correct, FPR is a real AI failure
- If LLM judges say H≥2 → human may have been too conservative

**Judges:** GPT-4o and deepseek-r1:8b rated all 32 JDs independently using the same 0–3 scale and instruction set given to the human annotator. All JDs masked (Company_XX).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('data/batch-results/llm_annotation_audit.json') as f:
    audit_data = json.load(f)

judge_keys = sorted({k for r in audit_data for k in r.get('labels', {})})
complete = [r for r in audit_data
            if all(r.get('labels', {}).get(j, {}).get('label') is not None for j in judge_keys)]
n = len(complete)

human_labels = np.array([r['human_relevance'] for r in complete])

def cohen_kappa(a, b, threshold=2):
    n_ = len(a)
    po = sum((1 if x>=threshold else 0) == (1 if y>=threshold else 0) for x,y in zip(a,b)) / n_
    pa = sum(1 for x in a if x>=threshold) / n_
    pb = sum(1 for y in b if y>=threshold) / n_
    pe = pa*pb + (1-pa)*(1-pb)
    return (po-pe)/(1-pe) if pe<1 else 0.0

def icc_2way(matrix):
    n_, k = matrix.shape
    grand = matrix.mean(); rows_m = matrix.mean(axis=1); cols_m = matrix.mean(axis=0)
    SSr = k*np.sum((rows_m-grand)**2); SSc = n_*np.sum((cols_m-grand)**2)
    SSe = np.sum((matrix - rows_m[:,None] - cols_m[None,:] + grand)**2)
    MSr=SSr/(n_-1); MSc=SSc/(k-1) if k>1 else 0; MSe=SSe/((n_-1)*(k-1)) if k>1 else 1e-9
    return float(max(0, (MSr-MSe)/(MSr+(k-1)*MSe+k*(MSc-MSe)/n_)))

# ── Agreement with human ─────────────────────────────────────────────────────
print(f'Annotation audit (n={n}) — Judge agreement with human labels')
print(f'(Binary: ≥2 = relevant)')
print()
print(f'{"Judge":<18}  {"r (Pearson)":>11}  {"Cohen κ":>8}  {"ICC":>7}')
print('─'*48)
for j in judge_keys:
    jlabels = np.array([r['labels'][j]['label'] for r in complete])
    r_val = float(np.corrcoef(human_labels, jlabels)[0,1])
    kappa = cohen_kappa(human_labels.tolist(), jlabels.tolist())
    icc = icc_2way(np.column_stack([human_labels, jlabels]).astype(float))
    print(f'{j:<18}  {r_val:>11.3f}  {kappa:>8.3f}  {icc:>7.3f}')

# Inter-judge
if len(judge_keys) >= 2:
    j1, j2 = judge_keys[0], judge_keys[1]
    l1 = np.array([r['labels'][j1]['label'] for r in complete])
    l2 = np.array([r['labels'][j2]['label'] for r in complete])
    inter_r = float(np.corrcoef(l1, l2)[0,1])
    inter_k = cohen_kappa(l1.tolist(), l2.tolist())
    inter_icc = icc_2way(np.column_stack([l1, l2]).astype(float))
    print()
    print(f'Inter-judge ({j1} vs {j2}):')
    print(f'  r={inter_r:.3f}  κ={inter_k:.3f}  ICC={inter_icc:.3f}')

# ── FP audit ─────────────────────────────────────────────────────────────────
fp_cases = [r for r in complete if r['human_relevance'] <= 1 and (r.get('v1_score') or 0) >= 50]
print(f'\nFP audit — {len(fp_cases)} cases (AI≥50, Human≤1):')
print(f'{"JD":<44} {"H":>2}  {"AI":>4}  ', end='')
for j in judge_keys: print(f'{j[:10]:>10}  ', end='')
print('Verdict')
print('─'*85)

llm_human = 0; llm_ai = 0
for r in sorted(fp_cases, key=lambda x: x.get('v1_score') or 0, reverse=True):
    jlabels = [r['labels'][j]['label'] for j in judge_keys]
    llm_mean = np.mean(jlabels)
    verdict = 'Human ✓' if llm_mean < 2 else 'AI right?'
    if llm_mean < 2: llm_human += 1
    else: llm_ai += 1
    print(f'{r["jd_id"][:43]:<44} {r["human_relevance"]:>2}  {r.get("v1_score") or 0:>4}  ', end='')
    for lbl in jlabels: print(f'{lbl:>10}  ', end='')
    print(verdict)

print(f'\nVerdict: {llm_human}/8 FPs confirmed as real AI failures  |  {llm_ai}/8 human label may be off')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Track D — LLM Annotation Audit', fontsize=13, fontweight='bold')

# Left: scatter human vs each judge
colors_j = ['#6366f1', '#f59e0b']
for ax, j, c in zip(axes, judge_keys, colors_j):
    jlabels = np.array([r['labels'][j]['label'] for r in complete])
    h_arr = human_labels + np.random.RandomState(42).uniform(-0.12, 0.12, len(human_labels))
    j_arr = jlabels     + np.random.RandomState(43).uniform(-0.12, 0.12, len(jlabels))
    ax.scatter(h_arr, j_arr, alpha=0.6, color=c, s=40)
    ax.plot([-0.5,3.5],[-0.5,3.5],'k--',alpha=0.3,lw=1)
    r_val = float(np.corrcoef(human_labels, jlabels)[0,1])
    ax.set_xlabel('Human label (0–3)'); ax.set_ylabel(f'{j} label (0–3)')
    ax.set_title(f'{j}  (r={r_val:.3f})')
    ax.set_xticks([0,1,2,3]); ax.set_yticks([0,1,2,3])
    ax.set_xlim(-0.5,3.5); ax.set_ylim(-0.5,3.5)

plt.tight_layout()
plt.savefig('results/figures/llm_annotation_audit.png', dpi=150, bbox_inches='tight')
plt.show()

---
### Part 16 — Findings

**LLM judges agree strongly with human labels** (κ=0.625 for GPT-4o, κ=0.562 for deepseek). Both judges' inter-agreement is even higher (κ=0.695), suggesting a consistent evaluation signal independent of the human annotator.

**FP verdict — 6/8 confirmed as real AI failures:**

| FP case | GPT-4o | deepseek | Verdict |
|---------|--------|----------|---------|
| Arrive People Analytics | 2 | 2 | Human too conservative — genuinely borderline |
| Atlantica Talent Analyst | 2 | 2 | Human too conservative — genuinely borderline |
| LinkedIn HR Data Eng | 1 | 1 | **Real FP** — human correct |
| QIC data analyst | 1 | 2 | **Real FP** (mixed signal) |
| Product Growth Analyst | 1 | 2 | **Real FP** (mixed signal) |
| Applied AI Engineer | 0 | 1 | **Real FP** — human correct |
| Digital Learning (Schneider) | 1 | 2 | **Real FP** (mixed signal) |
| Senior AI Engineer | 0 | 1 | **Real FP** — human correct |

**Implication:** The true FPR on unambiguously irrelevant JDs is closer to **37.5%** (6/16), not 50%. Two of the eight "false positives" were human annotation errors — those JDs are genuinely borderline cases where the AI was partially right.